In [ ]:
from dask.distributed import LocalCluster, Client
import dask.dataframe as dd
import pandas as pd
from rdkit import Chem

from smiles_blocks.files import PROCESSED_DATA_DIR


In [ ]:
cluster = LocalCluster(n_workers=10, threads_per_worker=1)
client = Client(cluster)

client

In [ ]:
data_dir = PROCESSED_DATA_DIR / "moses_repacked"

In [ ]:
ddf = dd.read_parquet(list(data_dir.glob("*.parquet")))

In [ ]:
def process_partition(df : pd.DataFrame) -> pd.DataFrame:

    # rename  columns to be consistent with the rest of the codebase
    df["smiles"] = df["SMILES"]
    df['split'] = df['SPLIT']
    df.drop(columns=["SMILES", "SPLIT"], inplace=True)
    
    df["unique_smiles"] = df["smiles"].apply(lambda x: Chem.MolToSmiles(Chem.MolFromSmiles(x), canonical=True, allBondsExplicit=True, kekuleSmiles=True))
 
    return df

In [ ]:
new_ddf = ddf.map_partitions(process_partition, meta={"smiles": "str", "split": "str", "unique_smiles": "str"})
new_ddf.to_parquet(PROCESSED_DATA_DIR / "moses_canonical_control", write_index=False)